# AFDM 第二步：CPP 与静态多径

在本 Notebook 中，先知道真实信道和正确帧起点，再验证接收检测器能否恢复已知比特。
这是纯 NumPy 软件实验，不加载 FPGA、不使用 ADC/DAC/DMA。
独立文件，上传后在新 Python 内核中运行全部单元格即可，不必先运行 02。

链路：QPSK → IDAFT → 添加 CPP → 静态多径线性卷积 → 去前缀 → DAFT → 已知信道 ZF/LMMSE 检测 → 比特比较。

CPP 与有效信道依据 [AFDM 原论文](https://arxiv.org/html/2204.12798v2) 式 (19)、(22)～(25)：
`s[n] = s[N+n] * exp(-j*2*pi*c1*(N*N + 2*N*n))`，其中 `n=-L,...,-1`。
N 为偶数且 2*N*c1 为整数时，CPP 退化为普通 CP。还会单独检查不满足此条件的参数，避免只验证特殊情况。

假设：整数时延、块间静态、完美定时、零频偏、真实信道已知。尚未实现多普勒、同步或信道估计。
这里的密集矩阵 ZF/LMMSE 是小规模正确性参考，不代表已完成 FPGA 可用的低复杂度检测器。


In [ ]:
import numpy as np

SEED = 2026111702
N = 64
BLOCKS = 256
L = 16
c1 = 1.0 / (2*N)
c2 = np.sqrt(2.0) / (N*N)
general_c1 = 0.37 / N
delays = np.array([0, 2, 5], dtype=int)
gains = np.array([1.0, 0.55*np.exp(0.4j), 0.25*np.exp(-0.8j)])
gains = gains / np.linalg.norm(gains)
SNR_DB = [0, 10, 20]
rng = np.random.default_rng(SEED)
print('NumPy:', np.__version__)
print('N =', N, ', CPP length =', L, ', blocks =', BLOCKS)
print('Path delays:', delays.tolist())
print('Path gains:', gains)
def idaft(symbols, c1, c2):
    """输入最后一维为 DAFT 符号，输出单位化的时域波形。"""
    symbols = np.asarray(symbols, dtype=np.complex128)
    n = np.arange(symbols.shape[-1])
    return np.exp(2j*np.pi*c1*n*n) * np.fft.ifft(
        symbols * np.exp(2j*np.pi*c2*n*n), axis=-1, norm='ortho')

def daft(samples, c1, c2):
    """输入时域采样，输出 DAFT 域符号。"""
    samples = np.asarray(samples, dtype=np.complex128)
    n = np.arange(samples.shape[-1])
    return np.exp(-2j*np.pi*c2*n*n) * np.fft.fft(
        samples * np.exp(-2j*np.pi*c1*n*n), axis=-1, norm='ortho')

def qpsk_map(bits):
    bits = np.asarray(bits)
    return ((1.0-2.0*bits[..., 0]) + 1j*(1.0-2.0*bits[..., 1])) / np.sqrt(2.0)

def qpsk_bits(symbols):
    return np.stack([symbols.real < 0, symbols.imag < 0], axis=-1).astype(np.uint8)


In [ ]:
def add_cpp(samples, length, chirp_c1):
    samples = np.asarray(samples, dtype=np.complex128)
    size = samples.shape[-1]
    if not isinstance(length, (int, np.integer)) or not 0 <= length <= size:
        raise ValueError('CPP length must be an integer between 0 and N')
    if length == 0:
        return samples.copy()
    negative_n = np.arange(-length, 0)
    phase = np.exp(-2j*np.pi*chirp_c1*(size*size + 2*size*negative_n))
    return np.concatenate([samples[..., -length:]*phase, samples], axis=-1)

def linear_channel(samples, path_delays, path_gains):
    """真正的线性卷积，不使用循环卷积替代物理通路。"""
    samples = np.asarray(samples, dtype=np.complex128)
    path_delays = np.asarray(path_delays)
    if path_delays.ndim != 1 or len(path_delays) == 0 or len(path_delays) != len(path_gains):
        raise ValueError('Channel paths are invalid')
    if not np.issubdtype(path_delays.dtype, np.integer) or np.any(path_delays < 0):
        raise ValueError('Integer nonnegative delays required')
    received = np.zeros(samples.shape[:-1]+(samples.shape[-1]+int(max(path_delays)),), dtype=np.complex128)
    for delay, gain in zip(path_delays, path_gains):
        received[..., delay:delay+samples.shape[-1]] += gain*samples
    return received

def receive_block(samples, length, chirp_c1, path_delays=delays, path_gains=gains):
    if length < max(path_delays):
        raise ValueError('CPP is shorter than maximum channel delay')
    full = linear_channel(add_cpp(samples, length, chirp_c1), path_delays, path_gains)
    return full[..., length:length+samples.shape[-1]]

def time_channel_matrix(size, chirp_c1, path_delays=delays, path_gains=gains):
    """用索引和前缀边界相位独立构造 H，与线性卷积输出对照。"""
    if max(path_delays) >= size:
        raise ValueError('Delays must be less than N')
    matrix = np.zeros((size, size), dtype=np.complex128)
    n = np.arange(size)
    for delay, gain in zip(path_delays, path_gains):
        k = n-delay
        phase = np.ones(size, dtype=np.complex128)
        mask = k < 0
        phase[mask] = np.exp(-2j*np.pi*chirp_c1*(size*size+2*size*k[mask]))
        matrix[n, k % size] += gain*phase
    return matrix

def daft_matrix(size, chirp_c1, chirp_c2):
    m = np.arange(size)[:, None]
    n = np.arange(size)[None, :]
    return np.exp(-2j*np.pi*(chirp_c2*m*m+m*n/size+chirp_c1*n*n))/np.sqrt(size)


## 先检查 CPP 及通道边界

In [ ]:
tx_bits = rng.integers(0, 2, size=(BLOCKS, N, 2), dtype=np.uint8)
symbols = qpsk_map(tx_bits)
wave = idaft(symbols, c1, c2)
cpp = add_cpp(wave, L, c1)
cp_error = float(np.max(np.abs(cpp[:, :L]-wave[:, -L:])))
assert cp_error < 1e-12
print('PASS 1: 满足特殊条件时 CPP 等于 CP，误差 =', cp_error)

general_wave = idaft(symbols[:4], general_c1, c2)
general_prefix = add_cpp(general_wave, L, general_c1)[:, :L]
# 直接在负时间索引求 IDAFT 基函数，独立检查 CPP 相位。
negative_n = np.arange(-L, 0)
m = np.arange(N)
negative_basis = np.exp(2j*np.pi*(general_c1*negative_n[:, None]**2
    + c2*m[None, :]**2 + negative_n[:, None]*m[None, :]/N))/np.sqrt(N)
prefix_reference = symbols[:4] @ negative_basis.T
general_prefix_error = float(np.max(np.abs(general_prefix-prefix_reference)))
assert general_prefix_error < 1e-11
assert np.max(np.abs(general_prefix-general_wave[:, -L:])) > 0.1
print('PASS 2: 一般 CPP 与负索引公式一致，误差 =', general_prefix_error)

assert np.array_equal(add_cpp(wave, 0, c1), wave)
for invalid_length in [-1, N+1, 1.5]:
    try:
        add_cpp(wave, invalid_length, c1)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid prefix length was accepted')
try:
    receive_block(wave, int(max(delays))-1, c1)
except ValueError:
    pass
else:
    raise AssertionError('Short prefix was accepted')
print('PASS 3: 零长度、非法长度与不足的前缀检查通过')


In [ ]:
modes = [('AFDM', c1, c2), ('AFDM-general-CPP', general_c1, c2), ('OFDM', 0.0, 0.0)]
mode_checks = []
for name, mode_c1, mode_c2 in modes:
    test_wave = idaft(symbols[:8], mode_c1, mode_c2)
    received = receive_block(test_wave, L, mode_c1)
    H = time_channel_matrix(N, mode_c1)
    channel_error = float(np.max(np.abs(received-test_wave @ H.T)))
    assert channel_error < 1e-11
    A = daft_matrix(N, mode_c1, mode_c2)
    H_eff = A @ H @ A.conj().T
    Y = daft(received, mode_c1, mode_c2)
    effective_error = float(np.max(np.abs(Y-symbols[:8] @ H_eff.T)))
    assert effective_error < 1e-10
    mode_checks.append(dict(mode=name, time_channel_error=channel_error, effective_channel_error=effective_error))
print('PASS 4: 三种模式的线性卷积与 H、H_eff 矩阵输出一致')

# 检查连续两块传输时，前一块的尾部不污染第二块有效采样。
for mode_c1 in [c1, general_c1, 0.0]:
    pair_wave = idaft(symbols[:2], mode_c1, c2)
    framed = add_cpp(pair_wave, L, mode_c1)
    stream = linear_channel(framed.reshape(-1), delays, gains)
    offset = N+L+L
    second = stream[offset:offset+N]
    isolated = receive_block(pair_wave[1], L, mode_c1)
    assert np.max(np.abs(second-isolated)) < 1e-11
print('PASS 5: 连续块边界测试通过，足够长的 CPP 隔离上一块尾部')

for row, (name, mode_c1, mode_c2) in zip(mode_checks, modes):
    transmitted = idaft(symbols, mode_c1, mode_c2)
    received = receive_block(transmitted, L, mode_c1)
    A = daft_matrix(N, mode_c1, mode_c2)
    H_eff = A @ time_channel_matrix(N, mode_c1) @ A.conj().T
    Y = daft(received, mode_c1, mode_c2)
    recovered = np.linalg.solve(H_eff, Y.T).T
    max_error = float(np.max(np.abs(recovered-symbols)))
    errors = int(np.count_nonzero(qpsk_bits(recovered) != tx_bits))
    assert max_error < 1e-10 and errors == 0
    row.update(symbol_recovery_error=max_error, bit_errors=errors, bits=int(tx_bits.size), condition_number=float(np.linalg.cond(H_eff)))
    print(name, ':', errors, '/', tx_bits.size, 'errors; symbol error =', max_error)
print('PASS 6: 三种模式无噪声、已知静态多径、ZF 检测后均为 0 错误')

A0 = daft_matrix(N, 0.0, 0.0)
ofdm_effective = A0 @ time_channel_matrix(N, 0.0) @ A0.conj().T
off_diagonal_error = float(np.max(np.abs(ofdm_effective-np.diag(np.diag(ofdm_effective)))))
assert off_diagonal_error < 1e-11
print('PASS 7: 静态多径下 OFDM 有效信道为对角矩阵，非对角误差 =', off_diagonal_error)


## 加入噪声后使用真实信道检测
ZF 用于无噪声正确性；带噪声时使用 LMMSE。两个矩阵都由真实信道参数给出，尚未从导频估计。

In [ ]:
# 使用真实信道 H_eff 的 LMMSE 检测，不是信道估计。
# Es=1；SNR 定义为加入前缀前的发射 Es/N0，复噪声方差为 1/SNR。
# 两种波形均使用长度 L 的前缀；这里不把前缀开销换算成 Eb/N0。
system = {}
for name, mode_c1, mode_c2 in [('AFDM', c1, c2), ('OFDM', 0.0, 0.0)]:
    transmitted = idaft(symbols, mode_c1, mode_c2)
    clean = receive_block(transmitted, L, mode_c1)
    A = daft_matrix(N, mode_c1, mode_c2)
    H_eff = A @ time_channel_matrix(N, mode_c1) @ A.conj().T
    system[name] = (mode_c1, mode_c2, clean, H_eff)

rows = []
print('SNR(dB)  AFDM_LMMSE错误/比特数  OFDM_LMMSE错误/比特数  AFDM_MSE  OFDM_MSE')
for snr_db in SNR_DB:
    variance = 10.0**(-snr_db/10.0)
    noise = np.sqrt(variance/2)*(rng.standard_normal((BLOCKS, N))+1j*rng.standard_normal((BLOCKS, N)))
    row = dict(snr_db=snr_db, bits=int(tx_bits.size), noise_variance=variance)
    for name, (mode_c1, mode_c2, clean, H_eff) in system.items():
        Y = daft(clean+noise, mode_c1, mode_c2)
        W = np.linalg.solve(H_eff.conj().T @ H_eff + variance*np.eye(N), H_eff.conj().T)
        detected = Y @ W.T
        errors = int(np.count_nonzero(qpsk_bits(detected) != tx_bits))
        row[name] = dict(errors=errors, ber=errors/tx_bits.size,
            mse=float(np.mean(np.abs(detected-symbols)**2)),
            uncorrected_errors=int(np.count_nonzero(qpsk_bits(Y) != tx_bits)))
    rows.append(row)
    print('%7d  %5d/%-8d         %5d/%-8d         %.6f  %.6f' % (
        snr_db, row['AFDM']['errors'], tx_bits.size, row['OFDM']['errors'], tx_bits.size,
        row['AFDM']['mse'], row['OFDM']['mse']))
for name in ['AFDM', 'OFDM']:
    assert rows[-1][name]['mse'] < rows[0][name]['mse']
    assert rows[-1][name]['errors'] < rows[-1][name]['uncorrected_errors']
print('PASS 8: 加噪声后的已知信道检测检查通过')
print('不能从这一组固定静态信道结果推断 AFDM 的多普勒优势。')


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import uuid

result_dir = Path.cwd() / ('afdm_cpp_results_' + uuid.uuid4().hex[:8])
result_dir.mkdir(exist_ok=False)
summary = dict(scope='software-only; static known channel; perfect timing; no Doppler, estimation or RF',
    host_clock=datetime.now().isoformat(), numpy_version=np.__version__, seed=SEED,
    n=N, blocks=BLOCKS, cpp_length=L, c1=c1, c2=float(c2), general_c1=general_c1,
    paths=[dict(delay=int(d), gain_real=float(h.real), gain_imag=float(h.imag)) for d,h in zip(delays,gains)],
    snr_definition='transmit Es/N0, excludes prefix energy overhead',
    cpp_equals_cp_error=cp_error, general_cpp_formula_error=general_prefix_error,
    mode_checks=mode_checks, ofdm_off_diagonal_error=off_diagonal_error, noisy_results=rows)
(result_dir/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
np.savez(result_dir/'reference_vectors.npz', bits=tx_bits[0], symbols=symbols[0],
    afdm_wave=wave[0], afdm_with_cpp=cpp[0],
    clean_received=receive_block(wave[0],L,c1), delays=delays, gains=gains, c1=c1, c2=c2, cpp_length=L)
print('结果保存位置：', result_dir)
print('本阶段：CPP 与已知静态多径软件自检完成。')
print('下一步：整数/分数多普勒的已知信道模型；信道估计与射频仍待实现。')
